# Practical 8: Remove Uninformative Features

## Aim
To identify and remove uninformative features from a dataset with a categorical target vector using mutual information.

## Objectives
- Create a classification dataset.
- Identify the categorical target variable.
- Calculate the importance of each feature.
- Identify uninformative features.
- Remove uninformative features.
- Create a final dataset containing informative features.

In [1]:
# Import Pandas for data handling
import pandas as pd

# Import mutual_info_classif for measuring feature information
from sklearn.feature_selection import mutual_info_classif

## 1. Create Dataset

We create a dataset containing useful and uninformative features.

Features:

- Study_Hours → useful
- Attendance → useful
- Previous_Marks → useful
- Student_ID → uninformative
- Random_Code → uninformative

The target variable is `Result`, which is categorical:

- Pass
- Fail

In [2]:
# Create sample student data
data = {
    "Study_Hours": [
        8, 2, 7, 3, 9, 4, 6, 1, 8, 2,
        7, 3, 9, 5, 6, 2, 8, 4, 7, 1
    ],

    "Attendance": [
        90, 60, 85, 65, 95, 70, 80, 55, 92, 62,
        88, 68, 96, 75, 82, 58, 91, 72, 86, 50
    ],

    "Previous_Marks": [
        85, 45, 80, 50, 90, 55, 75, 40, 88, 48,
        82, 52, 94, 68, 78, 42, 86, 60, 81, 35
    ],

    # This feature is only an ID and does not represent
    # useful information about the result
    "Student_ID": [
        101, 102, 103, 104, 105,
        106, 107, 108, 109, 110,
        111, 112, 113, 114, 115,
        116, 117, 118, 119, 120
    ],

    # Random/uninformative feature
    "Random_Code": [
        17, 43, 12, 88, 35,
        91, 24, 67, 53, 29,
        76, 14, 82, 41, 95,
        33, 60, 21, 73, 48
    ],

    # Categorical target
    "Result": [
        "Pass", "Fail", "Pass", "Fail", "Pass",
        "Fail", "Pass", "Fail", "Pass", "Fail",
        "Pass", "Fail", "Pass", "Pass", "Pass",
        "Fail", "Pass", "Pass", "Pass", "Fail"
    ]
}

# Create DataFrame
df = pd.DataFrame(data)

# Display dataset
print("========== ORIGINAL DATASET ==========")
print(df)

========== ORIGINAL DATASET ==========
    Study_Hours  Attendance  Previous_Marks  Student_ID  Random_Code Result
0             8          90              85         101           17   Pass
1             2          60              45         102           43   Fail
2             7          85              80         103           12   Pass
3             3          65              50         104           88   Fail
4             9          95              90         105           35   Pass
5             4          70              55         106           91   Fail
6             6          80              75         107           24   Pass
7             1          55              40         108           67   Fail
8             8          92              88         109           53   Pass
9             2          62              48         110           29   Fail
10            7          88              82         111           76   Pass
11            3          68              52      

## 2. Separate Features and Target

The `Result` column is our categorical target.

The remaining columns are input features.

We convert the categorical target into numerical values:

- Fail → 0
- Pass → 1

In [3]:
# Select input features
X = df.drop("Result", axis=1)

# Convert categorical target into numerical values
y = df["Result"].replace({
    "Fail": 0,
    "Pass": 1
})

# Display features
print("Features:")
print(X)

# Display target
print("\nCategorical Target Converted to Numerical:")
print(y)

Features:
    Study_Hours  Attendance  Previous_Marks  Student_ID  Random_Code
0             8          90              85         101           17
1             2          60              45         102           43
2             7          85              80         103           12
3             3          65              50         104           88
4             9          95              90         105           35
5             4          70              55         106           91
6             6          80              75         107           24
7             1          55              40         108           67
8             8          92              88         109           53
9             2          62              48         110           29
10            7          88              82         111           76
11            3          68              52         112           14
12            9          96              94         113           82
13            5         

C:\Users\dell\AppData\Local\Temp\ipykernel_12856\9831997.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = df["Result"].replace({


## 3. Calculate Feature Information

Mutual Information measures how much information each feature provides about the target variable.

A higher value means the feature contains more information about the target.

A value close to zero indicates that the feature provides little or no useful information.

In [4]:
# Calculate mutual information between each feature and target
mi_scores = mutual_info_classif(
    X,
    y,
    random_state=42
)

# Create a DataFrame containing feature importance scores
mi_df = pd.DataFrame({
    "Feature": X.columns,
    "Mutual_Information": mi_scores
})

# Sort features by mutual information
mi_df = mi_df.sort_values(
    by="Mutual_Information",
    ascending=False
)

# Display scores
print("========== FEATURE INFORMATION ==========")
print(mi_df)

========== FEATURE INFORMATION ==========
          Feature  Mutual_Information
2  Previous_Marks            0.627361
1      Attendance            0.584504
0     Study_Hours            0.565278
3      Student_ID            0.000000
4     Random_Code            0.000000


## 4. Identify Uninformative Features

Features with very low mutual information are considered uninformative.

We use a threshold of `0.01`.

Features with:

**Mutual Information < 0.01**

will be removed.

In [5]:
# Define threshold for removing uninformative features
threshold = 0.01

# Select informative features
informative_features = mi_df[
    mi_df["Mutual_Information"] >= threshold
]["Feature"].tolist()

# Select uninformative features
uninformative_features = mi_df[
    mi_df["Mutual_Information"] < threshold
]["Feature"].tolist()

# Display results
print("Informative Features:")
print(informative_features)

print("\nUninformative Features:")
print(uninformative_features)

Informative Features:
['Previous_Marks', 'Attendance', 'Study_Hours']

Uninformative Features:
['Student_ID', 'Random_Code']


## 5. Remove Uninformative Features

We create a new DataFrame containing only the informative features.

This reduces unnecessary data and can improve the efficiency of machine learning models.

In [6]:
# Create a new dataset containing only informative features
X_selected = X[informative_features]

# Display final selected dataset
print("========== DATA AFTER FEATURE SELECTION ==========")
print(X_selected)

# Display remaining features
print("\nRemaining Features:")
print(X_selected.columns.tolist())

========== DATA AFTER FEATURE SELECTION ==========
    Previous_Marks  Attendance  Study_Hours
0               85          90            8
1               45          60            2
2               80          85            7
3               50          65            3
4               90          95            9
5               55          70            4
6               75          80            6
7               40          55            1
8               88          92            8
9               48          62            2
10              82          88            7
11              52          68            3
12              94          96            9
13              68          75            5
14              78          82            6
15              42          58            2
16              86          91            8
17              60          72            4
18              81          86            7
19              35          50            1

Remaining Features:
['Pr

## Result

The mutual information method was successfully used to identify informative and uninformative features for a categorical target vector.

Features with very low mutual information were removed, while informative features were retained.

Thus, unnecessary features were successfully removed from the dataset.